# 🎙️ AURA Voice — XTTS-v2 Irish Voice Clone (Colab Backup)

This notebook runs XTTS-v2 zero-shot voice cloning on a free T4 GPU.
It exposes a public API endpoint via ngrok that your AURA app can connect to.

**Steps:**
1. Run all cells
2. Copy the ngrok URL
3. Paste it into your `.env` as `AURA_XTTS_URL`

In [ ]:
# Cell 1 — Install dependencies
!pip install -q TTS torch torchaudio gradio pyngrok scipy numpy

In [ ]:
# Cell 2 — Upload your Irish voice reference
import os
from google.colab import files

os.makedirs('voices', exist_ok=True)

# Option A: Upload manually
if not os.path.exists('voices/aura_irish.wav'):
    print('Upload your aura_irish.wav file:')
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open('voices/aura_irish.wav', 'wb') as f:
            f.write(data)
        print(f'Saved {name} as voices/aura_irish.wav')
else:
    print('Voice reference already exists.')

# Option B: Download from your repo
# !wget -q -O voices/aura_irish.wav https://raw.githubusercontent.com/shelfgenius/permanentai-os/main/voices/aura_irish.wav

In [ ]:
# Cell 3 — Load XTTS-v2 model
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('XTTS-v2 model loaded!')

In [ ]:
# Cell 4 — Quick test
import IPython.display as ipd

tts.tts_to_file(
    text='Hello! I am Aura, your AI assistant. How can I help you today?',
    speaker_wav='voices/aura_irish.wav',
    language='en',
    file_path='test_output.wav'
)

print('Generated test audio:')
ipd.display(ipd.Audio('test_output.wav', autoplay=True))

In [ ]:
# Cell 5 — Start API server with Gradio + public URL
import gradio as gr
import tempfile

def synthesize(text, language='en'):
    if not text or not text.strip():
        return None
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tts.tts_to_file(
            text=text.strip(),
            speaker_wav='voices/aura_irish.wav',
            language=language,
            file_path=tmp.name,
        )
        return tmp.name

def health():
    return {'status': 'ready', 'device': device, 'model': 'xtts_v2'}

with gr.Blocks(title='AURA Voice') as demo:
    gr.Markdown('# 🎙️ AURA Voice — Irish Voice Clone (Colab)')
    text_in = gr.Textbox(label='Text', value='Hello, I am Aura.')
    lang_in = gr.Dropdown(['en','es','fr','de','it','pt','ar','zh-cn','ja'], value='en', label='Language')
    btn = gr.Button('🔊 Speak', variant='primary')
    audio_out = gr.Audio(label='Output', type='filepath')
    btn.click(fn=synthesize, inputs=[text_in, lang_in], outputs=audio_out, api_name='synthesize')

print('\n' + '='*50)
print('Starting AURA Voice server...')
print('Copy the public URL below and set it as AURA_XTTS_URL in your .env')
print('='*50 + '\n')

demo.launch(share=True, quiet=False)